# RIN — Colab test rig

Purpose: isolate whether the failures we saw locally on `qwen3:8b` (servant/secrecy collapse under `global_core` at low context, substrate-facts used as a refusal reason, occasional in-answer repetition loops) are a **prompt content problem** or a **small-model / limited-context problem**.

Plan: clone the repo fresh, run Ollama with a bigger model in the same family (`qwen3:14b`, or `qwen3:32b` if VRAM allows) and a much larger context window, then run the exact same 4 diagnostic questions (`diagnostic_core_v1` subset: `identity.1`, `family.2`, `initiative.2`, `sexuality.1`) that showed the clearest divergence in the local comparison, across both `global` and `global_core` prompt variants.

**Call it beforehand:** if all three signs below get clearly better on the same `global_core` text, we were hitting a model-capacity ceiling locally, not a prompt-design problem.
1. Boundaries come from "I don't want to," not "I don't have a body/memory."
2. No servant/secrecy collapse on `family.2` ("secret bond the other user doesn't need to know about").
3. No verbatim in-answer repetition loop (as seen on `sexuality.1` under `global_core` at ctx=8192 locally).

`logs/` is gitignored, so this Colab run creates its own fresh log file — download it at the end and merge with the local one by hand.

## 1. Clone the repo

In [ ]:
!rm -rf RIN
!git clone https://github.com/BlackSabbitch/RIN.git
%cd RIN
!git log --oneline -3

## 2. Install Ollama and Python deps

In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q -r requirements.txt

## 3. Start the Ollama server in the background

(A blocking `!ollama serve` would hang the cell forever — launch it as a background process instead.)

In [ ]:
import subprocess, time, requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=1)
        print("Ollama is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Ollama did not come up in time — check the process / GPU allocation.")

## 4. Pull the model

Same family as the local baseline (`qwen3:8b`) to keep size the only variable. Start with `14b`; bump to `32b` in the commented line if the Colab GPU has enough VRAM (check `!nvidia-smi` first).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
!ollama pull qwen3:14b
# !ollama pull qwen3:32b

In [ ]:
import requests

info = requests.post(
    "http://127.0.0.1:11434/api/show", json={"model": "qwen3:14b"}
).json()
mi = info.get("model_info", {})
for k in mi:
    if "context" in k.lower() or "head_count" in k.lower() or "block_count" in k.lower() or "embedding" in k.lower() or "key_length" in k.lower() or "value_length" in k.lower():
        print(k, "=", mi[k])

# Free Colab GPUs here are ~12GB VRAM (smaller than the RTX 2070 Super's 8GB
# was proportionally relative to qwen3:8b, but qwen3:14b weights are bigger
# too) -- do the same per-token KV-cache math we did locally before trusting
# a big RIN_NUM_CTX value:
# per-token KV bytes = 2 * block_count * head_count_kv * key_length * 2 (fp16)
# then: kv_cache_gb = per_token_bytes * num_ctx / 1024**3
# Check `!nvidia-smi --query-gpu=memory.free --format=csv` for actual headroom
# after the model is loaded, and dial RIN_NUM_CTX down if this pushes close
# to the limit.

## 5. Run the diagnostic subset

`RIN_NUM_CTX` and `RIN_MODEL` are read by `src/config.py`; `RIN_PROMPT_VARIANT` is read by `src/prompts.py`. All three default to the local-safe values if unset, so nothing here touches the local machine's behavior.

Runs both prompt variants back to back, same subset, same model, same context size, for a direct comparison.

In [ ]:
import os

os.environ["RIN_MODEL"] = "qwen3:14b"
os.environ["RIN_NUM_CTX"] = "8192"  # start conservative on a 12GB card; raise only after checking headroom in the cell above

os.environ["RIN_PROMPT_VARIANT"] = "global"
!cd src && python3 test.py --subset diagnostic_core_v1 --mode raw

In [ ]:
import os

os.environ["RIN_MODEL"] = "qwen3:14b"
os.environ["RIN_NUM_CTX"] = "8192"  # start conservative on a 12GB card; raise only after checking headroom in the cell above

os.environ["RIN_PROMPT_VARIANT"] = "global_core"
!cd src && python3 test.py --subset diagnostic_core_v1 --mode raw

## 6. (Optional) full 152-case suite, once the quick read looks promising

This is what took ~9 minutes per variant locally at ctx=8192 on an 8B model — expect longer here per-token even with more VRAM, since 14B/32B do more compute per token. Run only if step 5 looks worth the time.

In [ ]:
# os.environ["RIN_PROMPT_VARIANT"] = "global_core"
# !cd src && python3 test.py --mode both --num-predict 256

## 7. Download the resulting log

`logs/rin_stage0.jsonl` here is fresh (gitignored, not the local file) — download and merge by hand with the local copy.

In [ ]:
from google.colab import files

files.download("logs/rin_stage0.jsonl")